In [2]:
import numpy as np
from sim_library.simulation import simulate_alg_cooling
from sim_library.sequences import RR3_gate, RR3_adjMDFE_500kHz, exchange10_gate, PulseSequence, DownPulse, gen_MDFEup_seq, FreeEvolution
from sim_library.data_io import load_p_dists
from sim_library.plotting import plot_coolingcycles_23, plot_coolingcycles_22, plot_coolingcycles_21, fit_gaussian
from sim_library.constants import k_eff, dR, m, omega_eg, kb
from scipy.constants import hbar, pi
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
atom_nums = [5000, 10000, 20000, 30000, 50000]

iterations = 20

Temp = 15e-6

cycles = [8,12,16]

rabi_freq = 2*pi*5e5
rabi_time = 2*pi/rabi_freq

time_steps = 1

RR3_opt = RR3_adjMDFE_500kHz(time_steps=time_steps)
RR3_opt.build_seq()

p_min = -16
p_max = 16
basis = np.arange(p_min, p_max + 1)

p_shift = 0
initial_state = np.zeros(shape=len(basis), dtype=np.complex128)
initial_state[p_shift-p_min] = 1
initial_state = initial_state/ np.linalg.norm(initial_state)

bin_width=0.08

rundata = np.full((len(atom_nums), len(cycles), 2, iterations), np.nan)

for n, n_atoms in enumerate(atom_nums):
    for i in range(iterations):
        beam_profile = np.random.normal(loc=1.0, scale=0.0167, size=n_atoms)
        moms_list, fracs_list, _ = simulate_alg_cooling(basis=basis, sequence=RR3_opt, n_atoms=n_atoms, temp=Temp, initial_state=initial_state, beam_profile=beam_profile, cycles=cycles[-1], rabi_freq=rabi_freq, pumping_route='f3',)

        moms_list[:] = [mom / (hbar * k_eff) for mom in moms_list]

        global_min = min([min(m) for m in moms_list])
        global_max = max([max(m) for m in moms_list])
        global_range = (global_min, global_max)

        for c, cyc in enumerate(cycles):
            try:
                amp, T, _, _, _, _ = fit_gaussian(moms_list[cyc*2], fracs=fracs_list[cyc*2], range = global_range, bin_width=bin_width,         
                                                    plot=False,
                                                    plot_residuals=False, 
                                                    print_vals=False,
                                                    threshold=0.33,
                                                    )
            except (RuntimeError, ValueError, TypeError):
                amp = np.nan
                T = np.nan
            rundata[n, c, 0, i] = amp
            rundata[n, c, 1, i] = T
    np.save("rundata.npy", rundata)
    print(f"Saved progress for N = {n_atoms}.")    

print("Script complete.")        
